# BeyondSmile: A Challenge on Detecting Depression through Facial Behavior and Head Gestures


In [1]:
import pandas as pd
import numpy as np
import tsfel
import neurokit2 as nk
import matplotlib.pyplot as plt
import datetime
import json
import pycatch22


# Read the colums for the data

In [2]:
columns_tself = pd.read_csv('./columns_mid_tsfel_prob_eye.csv')

In [3]:
del columns_tself['Unnamed: 0']

In [4]:
tself_columns_mid = list(columns_tself.columns)


In [5]:
columns_pycatch = pd.read_csv('./columns_mid_pycatch_prob_eye.csv')

In [6]:
del columns_pycatch['Unnamed: 0']

In [7]:
pycatch_columns_mid = list(columns_pycatch.columns)


In [8]:
tself_columns_mor = [name.replace('mid', 'mor') for name in tself_columns_mid]
tself_columns_aft = [name.replace('mid', 'aft') for name in tself_columns_mid]
tself_columns_eve = [name.replace('mid', 'eve') for name in tself_columns_mid]

In [9]:
pycatch_columns_mor = [name.replace('mid', 'mor') for name in pycatch_columns_mid]
pycatch_columns_aft = [name.replace('mid', 'aft') for name in pycatch_columns_mid]
pycatch_columns_eve = [name.replace('mid', 'eve') for name in pycatch_columns_mid]

In [10]:
data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')


# Read the labels


In [11]:
def closest_index(target_value, timeline_list):
    differences = np.abs(np.array(timeline_list) - target_value)
    closest_index = differences.argmin()

    return closest_index
    

In [13]:
for record in range(0, len(data_phq)):
    data_patient_depression = data_phq.loc[record]
    patient = data_phq.loc[record]['pid']
    diagnosis = data_phq.loc[record]['depression_episode']
    start_monitoring = data_patient_depression.start_ts
    end_monitoring = data_patient_depression.end_ts
    #Change the dates into the timestamp
    element_start = datetime.datetime(2022, int(start_monitoring.split('/')[0]), int(start_monitoring.split('/')[1]))
    timestamp_start = datetime.datetime.timestamp(element_start)
    element_end = datetime.datetime(2022, int(end_monitoring.split('/')[0]), int(end_monitoring.split('/')[1]))
    timestamp_end = datetime.datetime.timestamp(element_end)
    #Reorder the data
    with open('./dataset/data/'+patient+ '.json', 'r') as f:
            data = json.load(f)

    times = []
    numbers = []
    for i in range(0, len(data)):
        times.append(int(data[i]['timestamp'])/1000)
        numbers.append(i)
    min_value = datetime.datetime.fromtimestamp(min(times)).isoformat()
    max_value = datetime.datetime.fromtimestamp(max(times)).isoformat()

    b = enumerate(times)
    c = sorted(b, key = lambda i:i[1])
    times_index_primary = []

    for e in c:
        times_index_primary.append(e[0])

    sorted_times = sorted(times)
    # Reorder data json

    data2 = []
    for i in range(0, len(times_index_primary)):
        data2.append(data[times_index_primary[i]])

    times = []
    numbers = []
    for i in range(0, len(data2)):
        times.append(int(data2[i]['timestamp'])/1000)
        numbers.append(i)

    # Find the beginning of the record and end of the record
    counter_start = 0
    while(sorted_times[counter_start]<timestamp_start):
        counter_start +=1

    counter_end = counter_start
    for i in range(counter_start, len(sorted_times)):
        if sorted_times[counter_end]<=timestamp_end:
            counter_end +=1
        else:
            break
    counter_end = counter_end - 1

    #Select subdataset
    data3 = data2[counter_start:counter_end+1]

    timeline = []
    timelinedate = []
    for time in range(0, len(data3)):
        timeline.append(float(data3[time]['timestamp'])/1000)
        timelinedate.append(datetime.datetime.fromtimestamp(float(data3[time]['timestamp'])/1000))

    # Define separete subdata for the midningt, morning, afternoon and evening 
    start_index_list = []
    end_index_list = []

    start_index = 0

    for portion in range(0, 100):

        flag =0
        start_value = timeline[start_index]


        target_value = start_value +60*60*24 #1 day more

        end_index = closest_index(target_value, timeline) 
        end_value = timeline[end_index]




        if (end_value - start_value) > 60*60*24:
            flag=1
            start_index_list.append(start_index)
            end_index_list.append(end_index - 1)

          #  print(datetime.datetime.fromtimestamp(timeline[start_index]))
          #  print(datetime.datetime.fromtimestamp(timeline[end_index-1]))

            start_index = end_index
        else:

            if (end_index+1)<len(timeline):
                if (timeline[end_index+1] - start_value) > 60*60*24:
                    flag =1
                    start_index_list.append(start_index)
                    end_index_list.append(end_index)

                 #   print(datetime.datetime.fromtimestamp(timeline[start_index]))
                 #   print(datetime.datetime.fromtimestamp(timeline[end_index-1]))


                    start_index = end_index +1




        if flag ==0:
            break

    sub_data = pd.DataFrame(columns=['record', 'start_subrecord', 'end_subrecord'])
    sub_data['start_subrecord'] = start_index_list
    sub_data['end_subrecord'] = end_index_list
    sub_data['record'] = 0
    sub_data['diagnosis'] = diagnosis

    if (timeline[-1] - timeline[0])<=86400:
        sub_data['start_subrecord'] = [0]
        sub_data['end_subrecord']= [len(data3)]
        sub_data['record']= [0]
        sub_data['diagnosis']= [diagnosis]

    if len(sub_data)>0:
        sample = 0
    else:
        sample = -1

    if sample == 0:  
        start_index = sub_data.loc[sample]['start_subrecord']
        end_index = sub_data.loc[sample]['end_subrecord']
        data_test = data3[start_index:end_index+1]
        # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
        midnight_time = []
        morning_time = []
        afternoon_time = []
        evening_time = []
        for i in range(0, len(data_test)):
            hour_sample = datetime.datetime.fromtimestamp(float(data3[i]['timestamp'])/1000).hour
            if hour_sample>=0 and hour_sample<6:
                midnight_time.append(i)
            if hour_sample>=6 and hour_sample<12:
                morning_time.append(i)
            if hour_sample>=12 and hour_sample<18:
                afternoon_time.append(i)
            if hour_sample>=18 and hour_sample<=23:
                evening_time.append(i)

        data_midnight = []
        for i in range(0, len(midnight_time)):
            data_midnight.append(data_test[midnight_time[i]])

        data_morning = []
        for i in range(0, len(morning_time)):
            data_morning.append(data_test[morning_time[i]])

        data_afternoon = []
        for i in range(0, len(afternoon_time)):
            data_afternoon.append(data_test[afternoon_time[i]])

        data_evening = []
        for i in range(0, len(evening_time)):
            data_evening.append(data_test[evening_time[i]])

        # Extract midnight featues for smiling and open eyes probabilities
        COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
        COLUMN_NAMES_mid = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
        records_prob_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

        for i in range(0, len(data_midnight)):
            prob_data = data_midnight[i]['classification']
            prob_values = list(prob_data.values())

            if len(prob_data)!=0:
                records_prob_mid.loc[i] = prob_values
            else: 
                records_prob_mid.loc[i] = [np.nan]*3

        records_prob_mid_cleaned = records_prob_mid.copy()
        records_prob_mid_cleaned = records_prob_mid_cleaned.dropna()

        if len(records_prob_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)
            X_mid_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mid_to_delete.append(name)
            X = X.drop(X_mid_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mid)
            X.loc[0] = [np.nan]*372

        data_prob_mid = X.copy()

        if len(records_prob_mid_cleaned)>3:     
            for j in range(0, len(COLUMN_NAMES_mid)):
                name_prob = COLUMN_NAMES_mid[j]
                features_pycatch = pycatch22.catch22_all(records_prob_mid_cleaned[name_prob])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                features_prob_sub_mid = pd.DataFrame(columns=COLUMN)
                features_prob_sub_mid.loc[0] = features_pycatch['values']
                if j == 0:
                    features_prob_mid = features_prob_sub_mid.copy()
                else:
                    features_prob_mid = pd.concat([features_prob_mid, features_prob_sub_mid], axis=1)
        else:
            features_prob_mid = pd.DataFrame(columns=pycatch_columns_mid)
            features_prob_mid.loc[0] = [np.nan]*66 

        data_prob_mid = pd.concat([data_prob_mid, features_prob_mid], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_prob_mid_cleaned.columns]
        data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_prob_mid= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_prob_mid.append(approximate_entropy)
        data_approx_entropy_mid.loc[0] = app_ent_prob_mid

        data_prob_mid = pd.concat([data_prob_mid, data_approx_entropy_mid], axis=1)

        rsd_columns_mid = [name + '_rsd' for name in records_prob_mid_cleaned.columns]
        data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
        rsd_prob_mid = []
        for i in range(0, len(rsd_columns_mid)): 
            rsd = 100*np.std(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])/(np.mean(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_prob_mid.append(rsd)
        data_rsd_mid.loc[0] = rsd_prob_mid

        data_prob_mid = pd.concat([data_prob_mid, data_rsd_mid], axis=1)

        # For morning

        COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
        COLUMN_NAMES_mor = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
        records_prob_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

        for i in range(0, len(data_morning)):
            prob_data = data_morning[i]['classification']
            prob_values = list(prob_data.values())

            if len(prob_data)!=0:
                records_prob_mor.loc[i] = prob_values
            else: 
                records_prob_mor.loc[i] = [np.nan]*3


        records_prob_mor_cleaned = records_prob_mor.copy()
        records_prob_mor_cleaned = records_prob_mor_cleaned.dropna()

        if len(records_prob_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)
            X_mor_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_mor_to_delete.append(name)
            X = X.drop(X_mor_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_mor)
            X.loc[0] = [np.nan]*372

        data_prob_mor = X.copy()

        if len(records_prob_mor_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_mor)):
                name_prob = COLUMN_NAMES_mor[j]
                features_pycatch = pycatch22.catch22_all(records_prob_mor_cleaned[name_prob])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                features_prob_sub_mor = pd.DataFrame(columns=COLUMN)
                features_prob_sub_mor.loc[0] = features_pycatch['values']
                if j == 0:
                    features_prob_mor = features_prob_sub_mor.copy()
                else:
                    features_prob_mor = pd.concat([features_prob_mor, features_prob_sub_mor], axis=1)
        else:
            features_prob_mor = pd.DataFrame(columns=pycatch_columns_mor)
            features_prob_mor.loc[0] = [np.nan]*66 

        data_prob_mor = pd.concat([data_prob_mor, features_prob_mor], axis=1)
        approx_entropy_columns = [name + '_app_ent' for name in records_prob_mor_cleaned.columns]
        data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_prob_mor= []
        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_prob_mor.append(approximate_entropy)
        data_approx_entropy_mor.loc[0] = app_ent_prob_mor

        data_prob_mor = pd.concat([data_prob_mor, data_approx_entropy_mor], axis=1)

        rsd_columns_mor= [name + '_rsd' for name in records_prob_mor_cleaned.columns]
        data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
        rsd_prob_mor = []
        for i in range(0, len(rsd_columns_mor)): 
            rsd = 100*np.std(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])/(np.mean(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_prob_mor.append(rsd)
        data_rsd_mor.loc[0] = rsd_prob_mor

        data_prob_mor = pd.concat([data_prob_mor, data_rsd_mor], axis=1)

        # Afternoon data for smiling and eyes probabilities
        COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
        COLUMN_NAMES_aft = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
        records_prob_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

        for i in range(0, len(data_afternoon)):
            prob_data = data_afternoon[i]['classification']
            prob_values = list(prob_data.values())

            if len(prob_data)!=0:
                records_prob_aft.loc[i] = prob_values
            else: 
                records_prob_aft.loc[i] = [np.nan]*3

        records_prob_aft_cleaned = records_prob_aft.copy()
        records_prob_aft_cleaned = records_prob_aft_cleaned.dropna()

        if len(records_prob_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()
            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_prob_aft_cleaned)
            X_aft_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_aft_to_delete.append(name)
            X = X.drop(X_aft_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_aft)
            X.loc[0] = [np.nan]*372

        data_prob_aft = X.copy()

        if len(records_prob_aft_cleaned)>3:        
            for j in range(0, len(COLUMN_NAMES_aft)):
                name_prob = COLUMN_NAMES_aft[j]
                features_pycatch = pycatch22.catch22_all(records_prob_aft_cleaned[name_prob])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                features_prob_sub_aft = pd.DataFrame(columns=COLUMN)
                features_prob_sub_aft.loc[0] = features_pycatch['values']
                if j == 0:
                    features_prob_aft = features_prob_sub_aft.copy()
                else:
                    features_prob_aft = pd.concat([features_prob_aft, features_prob_sub_aft], axis=1)
        else:
            features_prob_aft = pd.DataFrame(columns=pycatch_columns_aft)
            features_prob_aft.loc[0] = [np.nan]*66 

        data_prob_aft = pd.concat([data_prob_aft, features_prob_aft], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_prob_aft_cleaned.columns]
        data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_prob_aft= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_prob_aft.append(approximate_entropy)
        data_approx_entropy_aft.loc[0] = app_ent_prob_aft

        data_prob_aft = pd.concat([data_prob_aft, data_approx_entropy_aft], axis=1)

        rsd_columns_aft= [name + '_aft' for name in records_prob_aft_cleaned.columns]
        data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
        rsd_prob_aft = []
        for i in range(0, len(rsd_columns_aft)): 
            rsd = 100*np.std(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])/(np.mean(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_prob_aft.append(rsd)
        data_rsd_aft.loc[0] = rsd_prob_aft

        data_prob_aft = pd.concat([data_prob_aft, data_rsd_aft], axis=1)

        # Probabilities for evening

        COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
        COLUMN_NAMES_eve = []
        for i in range(0, len(COLUMN_NAMES)):
            COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
        records_prob_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

        for i in range(0, len(data_evening)):
            prob_data = data_evening[i]['classification']
            prob_values = list(prob_data.values())

            if len(prob_data)!=0:
                records_prob_eve.loc[i] = prob_values
            else: 
                records_prob_eve.loc[i] = [np.nan]*3

        records_prob_eve_cleaned = records_prob_eve.copy()
        records_prob_eve_cleaned = records_prob_eve_cleaned.dropna()

        if len(records_prob_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
            cfg = tsfel.get_features_by_domain()

            # Extract features
            X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)
            X_eve_to_delete = []
            for name in X.columns:
                if 'Spectrogram mean coefficient_' in name:
                    X_eve_to_delete.append(name)
            X = X.drop(X_eve_to_delete, axis=1)

        else:
            X = pd.DataFrame(columns=tself_columns_eve)
            X.loc[0] = [np.nan]*372

        data_prob_eve = X.copy()

        if len(records_prob_eve_cleaned)>3:    
            for j in range(0, len(COLUMN_NAMES_eve)):
                name_prob = COLUMN_NAMES_eve[j]
                features_pycatch = pycatch22.catch22_all(records_prob_eve_cleaned[name_prob])
                COLUMN = []
                for i in range(0, len(features_pycatch['names'])):
                    COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                features_prob_sub_eve = pd.DataFrame(columns=COLUMN)
                features_prob_sub_eve.loc[0] = features_pycatch['values']
                if j == 0:
                    features_prob_eve = features_prob_sub_eve.copy()
                else:
                    features_prob_eve = pd.concat([features_prob_eve, features_prob_sub_eve], axis=1)
        else:
            features_prob_eve = pd.DataFrame(columns=pycatch_columns_eve)
            features_prob_eve.loc[0] = [np.nan]*66 

        data_prob_eve = pd.concat([data_prob_eve, features_prob_eve], axis=1)

        approx_entropy_columns = [name + '_app_ent' for name in records_prob_eve_cleaned.columns]
        data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
        app_ent_prob_eve= []

        for i in range(0, len(approx_entropy_columns)): 
            try:
                approximate_entropy, parameters = nk.entropy_approximate(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])
                 # Approximate entropy
            except:
                approximate_entropy = 0
            app_ent_prob_eve.append(approximate_entropy)
        data_approx_entropy_eve.loc[0] = app_ent_prob_eve

        data_prob_eve = pd.concat([data_prob_eve, data_approx_entropy_eve], axis=1)

        rsd_columns_eve= [name + '_rsd' for name in records_prob_eve_cleaned.columns]
        data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
        rsd_prob_eve = []
        for i in range(0, len(rsd_columns_eve)): 
            rsd = 100*np.std(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])/(np.mean(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])+0.00000000000000000000001)
            rsd_prob_eve.append(rsd)
        data_rsd_eve.loc[0] = rsd_prob_eve

        data_prob_eve = pd.concat([data_prob_eve, data_rsd_eve], axis=1)
        # Concat all probabilites
        data_prob = pd.concat([data_prob_mid, data_prob_mor, data_prob_aft, data_prob_eve], axis=1)
        information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data','subrecord'])
        information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'test', sample]
        data_prob = pd.concat([information_record, data_prob], axis=1, join='inner')

        data_all_prob = data_prob.copy()

    if len(sub_data)>1:
        flag_sub = 0
    else:
        flag_sub = -1

    if flag_sub!=-1:
        for sample in range(1, len(sub_data)):
            start_index = sub_data.loc[sample]['start_subrecord']
            end_index = sub_data.loc[sample]['end_subrecord']
            data_test = data3[start_index:end_index+1]
            # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
            midnight_time = []
            morning_time = []
            afternoon_time = []
            evening_time = []
            for i in range(0, len(data_test)):
                hour_sample = datetime.datetime.fromtimestamp(float(data3[i]['timestamp'])/1000).hour
                if hour_sample>=0 and hour_sample<6:
                    midnight_time.append(i)
                if hour_sample>=6 and hour_sample<12:
                    morning_time.append(i)
                if hour_sample>=12 and hour_sample<18:
                    afternoon_time.append(i)
                if hour_sample>=18 and hour_sample<=23:
                    evening_time.append(i)

            data_midnight = []
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_test[midnight_time[i]])

            data_morning = []
            for i in range(0, len(morning_time)):
                data_morning.append(data_test[morning_time[i]])

            data_afternoon = []
            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_test[afternoon_time[i]])

            data_evening = []
            for i in range(0, len(evening_time)):
                data_evening.append(data_test[evening_time[i]])

            # Extract midnight featues for smiling and open eyes probabilities
            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_mid = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
            records_prob_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

            for i in range(0, len(data_midnight)):
                prob_data = data_midnight[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_mid.loc[i] = prob_values
                else: 
                    records_prob_mid.loc[i] = [np.nan]*3

            records_prob_mid_cleaned = records_prob_mid.copy()
            records_prob_mid_cleaned = records_prob_mid_cleaned.dropna()

            if len(records_prob_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)
                X_mid_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mid_to_delete.append(name)
                X = X.drop(X_mid_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mid)
                X.loc[0] = [np.nan]*372

            data_prob_mid = X.copy()

            if len(records_prob_mid_cleaned)>3:     
                for j in range(0, len(COLUMN_NAMES_mid)):
                    name_prob = COLUMN_NAMES_mid[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_mid_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_mid = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_mid.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_mid = features_prob_sub_mid.copy()
                    else:
                        features_prob_mid = pd.concat([features_prob_mid, features_prob_sub_mid], axis=1)
            else:
                features_prob_mid = pd.DataFrame(columns=pycatch_columns_mid)
                features_prob_mid.loc[0] = [np.nan]*66 

            data_prob_mid = pd.concat([data_prob_mid, features_prob_mid], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_mid_cleaned.columns]
            data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_mid= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_mid.append(approximate_entropy)
            data_approx_entropy_mid.loc[0] = app_ent_prob_mid

            data_prob_mid = pd.concat([data_prob_mid, data_approx_entropy_mid], axis=1)

            rsd_columns_mid = [name + '_rsd' for name in records_prob_mid_cleaned.columns]
            data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
            rsd_prob_mid = []
            for i in range(0, len(rsd_columns_mid)): 
                rsd = 100*np.std(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])/(np.mean(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_mid.append(rsd)
            data_rsd_mid.loc[0] = rsd_prob_mid

            data_prob_mid = pd.concat([data_prob_mid, data_rsd_mid], axis=1)

            # For morning

            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_mor = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
            records_prob_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

            for i in range(0, len(data_morning)):
                prob_data = data_morning[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_mor.loc[i] = prob_values
                else: 
                    records_prob_mor.loc[i] = [np.nan]*3


            records_prob_mor_cleaned = records_prob_mor.copy()
            records_prob_mor_cleaned = records_prob_mor_cleaned.dropna()

            if len(records_prob_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)
                X_mor_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mor_to_delete.append(name)
                X = X.drop(X_mor_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mor)
                X.loc[0] = [np.nan]*372

            data_prob_mor = X.copy()

            if len(records_prob_mor_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mor)):
                    name_prob = COLUMN_NAMES_mor[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_mor_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_mor = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_mor.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_mor = features_prob_sub_mor.copy()
                    else:
                        features_prob_mor = pd.concat([features_prob_mor, features_prob_sub_mor], axis=1)
            else:
                features_prob_mor = pd.DataFrame(columns=pycatch_columns_mor)
                features_prob_mor.loc[0] = [np.nan]*66 

            data_prob_mor = pd.concat([data_prob_mor, features_prob_mor], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_prob_mor_cleaned.columns]
            data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_mor= []
            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_mor.append(approximate_entropy)
            data_approx_entropy_mor.loc[0] = app_ent_prob_mor

            data_prob_mor = pd.concat([data_prob_mor, data_approx_entropy_mor], axis=1)

            rsd_columns_mor= [name + '_rsd' for name in records_prob_mor_cleaned.columns]
            data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
            rsd_prob_mor = []
            for i in range(0, len(rsd_columns_mor)): 
                rsd = 100*np.std(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])/(np.mean(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_mor.append(rsd)
            data_rsd_mor.loc[0] = rsd_prob_mor

            data_prob_mor = pd.concat([data_prob_mor, data_rsd_mor], axis=1)

            # Afternoon data for smiling and eyes probabilities
            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_aft = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
            records_prob_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

            for i in range(0, len(data_afternoon)):
                prob_data = data_afternoon[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_aft.loc[i] = prob_values
                else: 
                    records_prob_aft.loc[i] = [np.nan]*3

            records_prob_aft_cleaned = records_prob_aft.copy()
            records_prob_aft_cleaned = records_prob_aft_cleaned.dropna()

            if len(records_prob_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()
                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_aft_cleaned)
                X_aft_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_aft_to_delete.append(name)
                X = X.drop(X_aft_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_aft)
                X.loc[0] = [np.nan]*372

            data_prob_aft = X.copy()

            if len(records_prob_aft_cleaned)>3:        
                for j in range(0, len(COLUMN_NAMES_aft)):
                    name_prob = COLUMN_NAMES_aft[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_aft_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_aft = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_aft.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_aft = features_prob_sub_aft.copy()
                    else:
                        features_prob_aft = pd.concat([features_prob_aft, features_prob_sub_aft], axis=1)
            else:
                features_prob_aft = pd.DataFrame(columns=pycatch_columns_aft)
                features_prob_aft.loc[0] = [np.nan]*66 

            data_prob_aft = pd.concat([data_prob_aft, features_prob_aft], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_aft_cleaned.columns]
            data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_aft= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_aft.append(approximate_entropy)
            data_approx_entropy_aft.loc[0] = app_ent_prob_aft

            data_prob_aft = pd.concat([data_prob_aft, data_approx_entropy_aft], axis=1)

            rsd_columns_aft= [name + '_aft' for name in records_prob_aft_cleaned.columns]
            data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
            rsd_prob_aft = []
            for i in range(0, len(rsd_columns_aft)): 
                rsd = 100*np.std(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])/(np.mean(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_aft.append(rsd)
            data_rsd_aft.loc[0] = rsd_prob_aft

            data_prob_aft = pd.concat([data_prob_aft, data_rsd_aft], axis=1)

            # Probabilities for evening

            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_eve = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
            records_prob_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

            for i in range(0, len(data_evening)):
                prob_data = data_evening[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_eve.loc[i] = prob_values
                else: 
                    records_prob_eve.loc[i] = [np.nan]*3

            records_prob_eve_cleaned = records_prob_eve.copy()
            records_prob_eve_cleaned = records_prob_eve_cleaned.dropna()

            if len(records_prob_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)
                X_eve_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_eve_to_delete.append(name)
                X = X.drop(X_eve_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_eve)
                X.loc[0] = [np.nan]*372

            data_prob_eve = X.copy()

            if len(records_prob_eve_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_eve)):
                    name_prob = COLUMN_NAMES_eve[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_eve_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_eve = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_eve.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_eve = features_prob_sub_eve.copy()
                    else:
                        features_prob_eve = pd.concat([features_prob_eve, features_prob_sub_eve], axis=1)
            else:
                features_prob_eve = pd.DataFrame(columns=pycatch_columns_eve)
                features_prob_eve.loc[0] = [np.nan]*66 

            data_prob_eve = pd.concat([data_prob_eve, features_prob_eve], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_eve_cleaned.columns]
            data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_eve= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_eve.append(approximate_entropy)
            data_approx_entropy_eve.loc[0] = app_ent_prob_eve

            data_prob_eve = pd.concat([data_prob_eve, data_approx_entropy_eve], axis=1)

            rsd_columns_eve= [name + '_rsd' for name in records_prob_eve_cleaned.columns]
            data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
            rsd_prob_eve = []
            for i in range(0, len(rsd_columns_eve)): 
                rsd = 100*np.std(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])/(np.mean(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_eve.append(rsd)
            data_rsd_eve.loc[0] = rsd_prob_eve

            data_prob_eve = pd.concat([data_prob_eve, data_rsd_eve], axis=1)
            # Concat all probabilites
            data_prob = pd.concat([data_prob_mid, data_prob_mor, data_prob_aft, data_prob_eve], axis=1)
            information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data','subrecord'])
            information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'test', sample]
            data_prob = pd.concat([information_record, data_prob], axis=1, join='inner')

            data_all_prob = pd.concat([data_all_prob, data_prob])

            start_index = 0
            end_index = sub_data.loc[sample]['start_subrecord']
            data_train = data3[start_index:end_index+1]
            # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
            midnight_time = []
            morning_time = []
            afternoon_time = []
            evening_time = []
            for i in range(0, len(data_train)):
                hour_sample = datetime.datetime.fromtimestamp(float(data3[i]['timestamp'])/1000).hour
                if hour_sample>=0 and hour_sample<6:
                    midnight_time.append(i)
                if hour_sample>=6 and hour_sample<12:
                    morning_time.append(i)
                if hour_sample>=12 and hour_sample<18:
                    afternoon_time.append(i)
                if hour_sample>=18 and hour_sample<=23:
                    evening_time.append(i)

            data_midnight = []
            for i in range(0, len(midnight_time)):
                data_midnight.append(data_train[midnight_time[i]])

            data_morning = []
            for i in range(0, len(morning_time)):
                data_morning.append(data_train[morning_time[i]])

            data_afternoon = []
            for i in range(0, len(afternoon_time)):
                data_afternoon.append(data_train[afternoon_time[i]])

            data_evening = []
            for i in range(0, len(evening_time)):
                data_evening.append(data_train[evening_time[i]])

            # Extract midnight featues for smiling and open eyes probabilities
            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_mid = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
            records_prob_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

            for i in range(0, len(data_midnight)):
                prob_data = data_midnight[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_mid.loc[i] = prob_values
                else: 
                    records_prob_mid.loc[i] = [np.nan]*3

            records_prob_mid_cleaned = records_prob_mid.copy()
            records_prob_mid_cleaned = records_prob_mid_cleaned.dropna()

            if len(records_prob_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)
                X_mid_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mid_to_delete.append(name)
                X = X.drop(X_mid_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mid)
                X.loc[0] = [np.nan]*372

            data_prob_mid = X.copy()

            if len(records_prob_mid_cleaned)>3:     
                for j in range(0, len(COLUMN_NAMES_mid)):
                    name_prob = COLUMN_NAMES_mid[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_mid_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_mid = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_mid.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_mid = features_prob_sub_mid.copy()
                    else:
                        features_prob_mid = pd.concat([features_prob_mid, features_prob_sub_mid], axis=1)
            else:
                features_prob_mid = pd.DataFrame(columns=pycatch_columns_mid)
                features_prob_mid.loc[0] = [np.nan]*66 

            data_prob_mid = pd.concat([data_prob_mid, features_prob_mid], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_mid_cleaned.columns]
            data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_mid= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_mid.append(approximate_entropy)
            data_approx_entropy_mid.loc[0] = app_ent_prob_mid

            data_prob_mid = pd.concat([data_prob_mid, data_approx_entropy_mid], axis=1)

            rsd_columns_mid = [name + '_rsd' for name in records_prob_mid_cleaned.columns]
            data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
            rsd_prob_mid = []
            for i in range(0, len(rsd_columns_mid)): 
                rsd = 100*np.std(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])/(np.mean(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_mid.append(rsd)
            data_rsd_mid.loc[0] = rsd_prob_mid

            data_prob_mid = pd.concat([data_prob_mid, data_rsd_mid], axis=1)

            # For morning

            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_mor = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
            records_prob_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

            for i in range(0, len(data_morning)):
                prob_data = data_morning[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_mor.loc[i] = prob_values
                else: 
                    records_prob_mor.loc[i] = [np.nan]*3


            records_prob_mor_cleaned = records_prob_mor.copy()
            records_prob_mor_cleaned = records_prob_mor_cleaned.dropna()

            if len(records_prob_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)
                X_mor_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_mor_to_delete.append(name)
                X = X.drop(X_mor_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_mor)
                X.loc[0] = [np.nan]*372

            data_prob_mor = X.copy()

            if len(records_prob_mor_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_mor)):
                    name_prob = COLUMN_NAMES_mor[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_mor_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_mor = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_mor.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_mor = features_prob_sub_mor.copy()
                    else:
                        features_prob_mor = pd.concat([features_prob_mor, features_prob_sub_mor], axis=1)
            else:
                features_prob_mor = pd.DataFrame(columns=pycatch_columns_mor)
                features_prob_mor.loc[0] = [np.nan]*66 

            data_prob_mor = pd.concat([data_prob_mor, features_prob_mor], axis=1)
            approx_entropy_columns = [name + '_app_ent' for name in records_prob_mor_cleaned.columns]
            data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_mor= []
            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_mor.append(approximate_entropy)
            data_approx_entropy_mor.loc[0] = app_ent_prob_mor

            data_prob_mor = pd.concat([data_prob_mor, data_approx_entropy_mor], axis=1)

            rsd_columns_mor= [name + '_rsd' for name in records_prob_mor_cleaned.columns]
            data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
            rsd_prob_mor = []
            for i in range(0, len(rsd_columns_mor)): 
                rsd = 100*np.std(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])/(np.mean(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_mor.append(rsd)
            data_rsd_mor.loc[0] = rsd_prob_mor

            data_prob_mor = pd.concat([data_prob_mor, data_rsd_mor], axis=1)

            # Afternoon data for smiling and eyes probabilities
            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_aft = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
            records_prob_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

            for i in range(0, len(data_afternoon)):
                prob_data = data_afternoon[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_aft.loc[i] = prob_values
                else: 
                    records_prob_aft.loc[i] = [np.nan]*3

            records_prob_aft_cleaned = records_prob_aft.copy()
            records_prob_aft_cleaned = records_prob_aft_cleaned.dropna()

            if len(records_prob_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()
                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_aft_cleaned)
                X_aft_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_aft_to_delete.append(name)
                X = X.drop(X_aft_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_aft)
                X.loc[0] = [np.nan]*372

            data_prob_aft = X.copy()

            if len(records_prob_aft_cleaned)>3:        
                for j in range(0, len(COLUMN_NAMES_aft)):
                    name_prob = COLUMN_NAMES_aft[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_aft_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_aft = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_aft.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_aft = features_prob_sub_aft.copy()
                    else:
                        features_prob_aft = pd.concat([features_prob_aft, features_prob_sub_aft], axis=1)
            else:
                features_prob_aft = pd.DataFrame(columns=pycatch_columns_aft)
                features_prob_aft.loc[0] = [np.nan]*66 

            data_prob_aft = pd.concat([data_prob_aft, features_prob_aft], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_aft_cleaned.columns]
            data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_aft= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_aft.append(approximate_entropy)
            data_approx_entropy_aft.loc[0] = app_ent_prob_aft

            data_prob_aft = pd.concat([data_prob_aft, data_approx_entropy_aft], axis=1)

            rsd_columns_aft= [name + '_aft' for name in records_prob_aft_cleaned.columns]
            data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
            rsd_prob_aft = []
            for i in range(0, len(rsd_columns_aft)): 
                rsd = 100*np.std(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])/(np.mean(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_aft.append(rsd)
            data_rsd_aft.loc[0] = rsd_prob_aft

            data_prob_aft = pd.concat([data_prob_aft, data_rsd_aft], axis=1)

            # Probabilities for evening

            COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
            COLUMN_NAMES_eve = []
            for i in range(0, len(COLUMN_NAMES)):
                COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
            records_prob_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

            for i in range(0, len(data_evening)):
                prob_data = data_evening[i]['classification']
                prob_values = list(prob_data.values())

                if len(prob_data)!=0:
                    records_prob_eve.loc[i] = prob_values
                else: 
                    records_prob_eve.loc[i] = [np.nan]*3

            records_prob_eve_cleaned = records_prob_eve.copy()
            records_prob_eve_cleaned = records_prob_eve_cleaned.dropna()

            if len(records_prob_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
                cfg = tsfel.get_features_by_domain()

                # Extract features
                X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)
                X_eve_to_delete = []
                for name in X.columns:
                    if 'Spectrogram mean coefficient_' in name:
                        X_eve_to_delete.append(name)
                X = X.drop(X_eve_to_delete, axis=1)

            else:
                X = pd.DataFrame(columns=tself_columns_eve)
                X.loc[0] = [np.nan]*372

            data_prob_eve = X.copy()

            if len(records_prob_eve_cleaned)>3:    
                for j in range(0, len(COLUMN_NAMES_eve)):
                    name_prob = COLUMN_NAMES_eve[j]
                    features_pycatch = pycatch22.catch22_all(records_prob_eve_cleaned[name_prob])
                    COLUMN = []
                    for i in range(0, len(features_pycatch['names'])):
                        COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
                    features_prob_sub_eve = pd.DataFrame(columns=COLUMN)
                    features_prob_sub_eve.loc[0] = features_pycatch['values']
                    if j == 0:
                        features_prob_eve = features_prob_sub_eve.copy()
                    else:
                        features_prob_eve = pd.concat([features_prob_eve, features_prob_sub_eve], axis=1)
            else:
                features_prob_eve = pd.DataFrame(columns=pycatch_columns_eve)
                features_prob_eve.loc[0] = [np.nan]*66 

            data_prob_eve = pd.concat([data_prob_eve, features_prob_eve], axis=1)

            approx_entropy_columns = [name + '_app_ent' for name in records_prob_eve_cleaned.columns]
            data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
            app_ent_prob_eve= []

            for i in range(0, len(approx_entropy_columns)): 
                try:
                    approximate_entropy, parameters = nk.entropy_approximate(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])
                     # Approximate entropy
                except:
                    approximate_entropy = 0
                app_ent_prob_eve.append(approximate_entropy)
            data_approx_entropy_eve.loc[0] = app_ent_prob_eve

            data_prob_eve = pd.concat([data_prob_eve, data_approx_entropy_eve], axis=1)

            rsd_columns_eve= [name + '_rsd' for name in records_prob_eve_cleaned.columns]
            data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
            rsd_prob_eve = []
            for i in range(0, len(rsd_columns_eve)): 
                rsd = 100*np.std(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])/(np.mean(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])+0.00000000000000000000001)
                rsd_prob_eve.append(rsd)
            data_rsd_eve.loc[0] = rsd_prob_eve

            data_prob_eve = pd.concat([data_prob_eve, data_rsd_eve], axis=1)
            # Concat all probabilites
            data_prob = pd.concat([data_prob_mid, data_prob_mor, data_prob_aft, data_prob_eve], axis=1)
            information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end', 'type_data','subrecord'])
            information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end, 'train', sample]
            data_prob = pd.concat([information_record, data_prob], axis=1, join='inner')

            data_all_prob= pd.concat([data_all_prob, data_prob])

    data_all_prob.to_csv('dataset/prob_cross/prob_'+str(record)+'_.csv')

C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:191: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:273: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:560: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:642: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:922: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)


...

C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:800: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:922: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:1004: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:1081: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_aft_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_18748\130156354.py:1162: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)


In [156]:
data_all_prob

,patient,record,diagnosis,timestamp_start,timestamp_end,type_data,subrecord,lefteye_mid_Absolute energy,lefteye_mid_Area under the curve,lefteye_mid_Autocorrelation,...,smiling_eve_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,smiling_eve_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,smiling_eve_SP_Summaries_welch_rect_centroid,smiling_eve_FC_LocalSimple_mean3_stderr,lefteye_eve_app_ent,righteye_eve_app_ent,smiling_eve_app_ent,lefteye_eve_rsd,righteye_eve_rsd,smiling_eve_rsd
0,P08,0,0,1.658354e+09,1.659996e+09,test,0,13.955394,0.134858,1.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,test,1,10.798848,0.117876,1.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,train,1,13.955394,0.134858,1.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,test,2,13.611254,0.130201,1.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,train,2,13.955394,0.134858,1.0,...,0.000000,0.000000,2.356194,0.180316,0.058892,0.287682,0.287682,33.767523,1.672398,122.974929
0,P08,0,0,1.658354e+09,1.659996e+09,test,3,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,train,3,13.955394,0.134858,1.0,...,0.000000,0.000000,2.356194,0.180316,0.058892,0.287682,0.287682,33.767523,1.672398,122.974929
0,P08,0,0,1.658354e+09,1.659996e+09,test,4,14.604941,0.138048,2.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
0,P08,0,0,1.658354e+09,1.659996e+09,train,4,13.955394,0.134858,1.0,...,0.000000,0.000000,2.356194,0.180316,0.058892,0.287682,0.287682,33.767523,1.672398,122.974929
0,P08,0,0,1.658354e+09,1.659996e+09,test,5,14.359278,0.136820,1.0,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
